# scratch256_lr3e3 — upper-width scaling control: FROM SCRATCH 10b x 256ch, 40 epochs, UNWEIGHTED r2_bulk (12.51M rows). Objective and validation are internally matched controls, but raw val loss is informational only. Gate stays GAMEPLAY.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v5.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v5.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/r2_bulk.pt.gz /content/r2_bulk.pt.gz
gz=os.path.getsize('/content/r2_bulk.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 740_453_101, f'.gz truncated! got {gz}; re-upload r2_bulk.pt.gz'
!gunzip -t /content/r2_bulk.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/r2_bulk.pt.gz > /content/alphatrain/data/r2_bulk.pt
pt=os.path.getsize('/content/alphatrain/data/r2_bulk.pt')
assert pt == 2_039_621_645, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 12,512,995 rows ({time.time()-t0:.0f}s)')
!rm /content/r2_bulk.pt.gz
!pip install -q numpy numba scipy

# Refuse the historical corrupt D4 implementation.
_dataset_source = open('/content/alphatrain/dataset.py').read()
assert ('def _build_line_direction_luts' in _dataset_source and
        'def _transform_observation' in _dataset_source), (
    'STALE CODE TARBALL: directional H/V/D1/D2 channels are not '
    'transformed under D4; rebuild/upload colorlines_pillar3d_v5.tar.gz.')
print('Exact directional-channel D4 transform confirmed.')


In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (scratch256_lr3e3) =====
NUM_BLOCKS = 10
CHANNELS = 256
EPOCHS   = 40
BATCH    = 32768
LR       = 3e-3
T        = 1.0
DW       = 0
BLEND    = 0.0
GAMMA    = 0  # unweighted — clean old-regime control
SEED     = 42
SAVE_STEPS = 1000
RUN      = "scratch256_lr3e3"
print(f'RUN={RUN} ep={EPOCHS} bs={BATCH} lr={LR} gamma={GAMMA}')


In [ ]:
# Resume from the last COMPLETED epoch if Colab stopped.
LATEST = (f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/latest.pt')
RESUME_ARGS = f'--resume {LATEST}' if os.path.exists(LATEST) else ''
print('Resume:', LATEST if RESUME_ARGS else 'none (new run)')
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    {RESUME_ARGS} --tensor-file alphatrain/data/r2_bulk.pt \
    --num-blocks {NUM_BLOCKS} --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} --disagree-gamma {GAMMA} \
    --save-every-steps {SAVE_STEPS} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/drive/MyDrive/alphatrain/{RUN}_ckpts 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (floor-first, 5k decides).


In [ ]:
# Checkpoints already save DIRECTLY to Drive ({RUN}_ckpts/) —
# every epoch_N.pt and e{E}_s{S}.pt lands there as it is written.
import glob
for f in sorted(glob.glob(f'/content/drive/MyDrive/alphatrain/{RUN}_ckpts/*.pt')):
    print(f)


## Evaluation protocol (local M5, C++ FP16)

Download approximately every third epoch first (3/6/9/12/18/24/30/36/40).
Convert each checkpoint directly to its own TorchScript path:

```bash
source .venv/bin/activate
python -m alphatrain.inference_cpp.export_ts \
  --model alphatrain/data/scratch256_lr3e3_ckpts_epoch_12.pt \
  --output alphatrain/inference_cpp/data/scratch256_lr3e3_ckpts_epoch_12_ts.pt
```

Run aggregate-only 999-game screens; per-game traces remain off by default:

```bash
cd alphatrain/inference_cpp
caffeinate -i -s ./build/eval \
  --model data/scratch256_lr3e3_ckpts_epoch_12_ts.pt --device mps --batch 500 \
  --seed-start 775000 --seed-end 775999 \
  --scores-out data/scratch256_lr3e3_ep12_775k_999.csv
```

Take only the best apparent plateau checkpoint(s) to 5,000 games by changing
`--seed-end` to `780000`. Compare aggregate distributions independently: P5/P10,
`<1000`, median, upper tail, and survival horizons. Matching seed identifiers are
environmental controls, never a paired-game statistic. Reserved final seeds
1,100,000--1,119,999 remain untouched.
